# Track A Supervised Classification Evidence Notebook

## Purpose
This notebook is a read-only evidence and review layer for Track A. It does not train models, recompute predictions, or regenerate governed artifacts.


## 1. Runtime Detection
Detect local vs Colab runtime, Python version, repository root, and safe device availability. This notebook is not the source of truth; governed artifacts, registries, and the frontend-ready JSON bundle are the source of truth.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import platform
import sys
from typing import Any

try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "inspection_ai").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("Unable to resolve repository root containing src/inspection_ai and configs/.")


REPO_ROOT = find_repo_root()
SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

try:
    import inspection_ai  # noqa: F401
    SRC_IMPORTABLE = True
except ImportError as exc:
    raise RuntimeError("src/ is not importable; governed project code cannot be referenced.") from exc

DEVICE_SUMMARY = {"torch_available": False, "device_hint": "unavailable"}
try:
    import torch
    DEVICE_SUMMARY = {
        "torch_available": True,
        "cuda_available": bool(torch.cuda.is_available()),
        "device_hint": "cuda" if torch.cuda.is_available() else "cpu",
    }
except Exception as exc:
    DEVICE_SUMMARY = {"torch_available": False, "device_hint": f"unavailable: {type(exc).__name__}"}

runtime_summary = {
    "runtime": "colab" if IN_COLAB else "local",
    "python": platform.python_version(),
    "repo_root": str(REPO_ROOT),
    "src_importable": SRC_IMPORTABLE,
    "device_summary": DEVICE_SUMMARY,
}
runtime_summary

## 2. Path And Artifact Resolution
Resolve paths from the repository root. Required evidence missing from this section should be treated as a notebook setup issue, not a model-quality issue.


In [ ]:
from dataclasses import dataclass


def repo_path(path: str | Path) -> Path:
    candidate = Path(path)
    if candidate.is_absolute():
        return candidate
    return REPO_ROOT / candidate


@dataclass(frozen=True)
class ArtifactRef:
    key: str
    path: str
    required: bool
    description: str


TRACK_A_RUNS = {
    "mlp": "c3b74a5c-34ce-4654-9e5d-915c886ec9f6",
    "cnn": "50993bc0-4dcd-48f3-a080-d6cbaf21d804",
    "resnet18": "1bc92561-c5bf-48f2-8246-b8f3d5718ffe",
}

HISTORICAL_NONCANONICAL_RUNS = {
    "resnet18_previous_comparison": "86eae01e-5913-4a91-8167-916f221fcb39",
}

ARTIFACTS: list[ArtifactRef] = [
    ArtifactRef("training_mlp", "artifacts/models/analysis/training_results/training_result__c3b74a5c-34ce-4654-9e5d-915c886ec9f6.json", True, "MLP TrainingResult"),
    ArtifactRef("training_cnn", "artifacts/models/analysis/training_results/training_result__50993bc0-4dcd-48f3-a080-d6cbaf21d804.json", True, "CNN TrainingResult"),
    ArtifactRef("training_resnet18", "artifacts/models/analysis/training_results/training_result__1bc92561-c5bf-48f2-8246-b8f3d5718ffe.json", True, "ResNet18 TrainingResult"),
    ArtifactRef("validation_mlp", "artifacts/models/metrics/classification_validation_evaluation__c3b74a5c-34ce-4654-9e5d-915c886ec9f6.json", True, "MLP validation evaluation"),
    ArtifactRef("validation_cnn", "artifacts/models/metrics/classification_validation_evaluation__50993bc0-4dcd-48f3-a080-d6cbaf21d804.json", True, "CNN validation evaluation"),
    ArtifactRef("validation_resnet18", "artifacts/models/metrics/classification_validation_evaluation__1bc92561-c5bf-48f2-8246-b8f3d5718ffe.json", True, "ResNet18 validation evaluation"),
    ArtifactRef("prediction_resnet18", "artifacts/models/error_analysis/track_a_resnet18_0.4.0_prediction_level_analysis__1bc92561-c5bf-48f2-8246-b8f3d5718ffe__validation.json", True, "ResNet18 prediction-level analysis"),
    ArtifactRef("threshold_resnet18", "artifacts/models/error_analysis/track_a_threshold_analysis__1bc92561-c5bf-48f2-8246-b8f3d5718ffe__validation.json", True, "ResNet18 threshold analysis"),
    ArtifactRef("quality_resnet18", "artifacts/models/analysis/track_a_resnet18_v0_4_0_quality_decision__1bc92561-c5bf-48f2-8246-b8f3d5718ffe.json", True, "ResNet18 quality decision"),
    ArtifactRef("metadata_resnet18", "artifacts/models/metadata/track_a_resnet18_v0_4_0_metadata_summary__1bc92561-c5bf-48f2-8246-b8f3d5718ffe.json", True, "ResNet18 metadata summary"),
    ArtifactRef("inventory_resnet18", "artifacts/models/inventory/track_a_resnet18_v0_4_0_artifact_inventory__1bc92561-c5bf-48f2-8246-b8f3d5718ffe.json", True, "ResNet18 artifact inventory"),
    ArtifactRef("comparison_current", "artifacts/models/comparisons/track_a_supervised_classification__c3b74a5c-34ce-4654-9e5d-915c886ec9f6__50993bc0-4dcd-48f3-a080-d6cbaf21d804__1bc92561-c5bf-48f2-8246-b8f3d5718ffe.json", True, "Current Track A comparison"),
    ArtifactRef("frontend_model_comparison_table", "artifacts/frontend/track_a/model_comparison_table.json", True, "Track A frontend model comparison table"),
    ArtifactRef("frontend_metric_cards", "artifacts/frontend/track_a/metric_cards.json", True, "Track A frontend metric cards"),
    ArtifactRef("frontend_confusion_matrix_chart_data", "artifacts/frontend/track_a/confusion_matrix_chart_data.json", True, "Track A frontend confusion matrix chart data"),
    ArtifactRef("frontend_threshold_curve_chart_data", "artifacts/frontend/track_a/threshold_curve_chart_data.json", True, "Track A frontend threshold curve chart data"),
    ArtifactRef("frontend_per_class_bar_chart_data", "artifacts/frontend/track_a/per_class_bar_chart_data.json", True, "Track A frontend per-class bar chart data"),
    ArtifactRef("frontend_error_distribution_pie_data", "artifacts/frontend/track_a/error_distribution_pie_data.json", True, "Track A frontend error distribution pie data"),
    ArtifactRef("frontend_sample_predictions_gallery", "artifacts/frontend/track_a/sample_predictions_gallery.json", True, "Track A frontend sample predictions gallery"),
    ArtifactRef("frontend_quality_decision_summary", "artifacts/frontend/track_a/quality_decision_summary.json", True, "Track A frontend quality decision summary"),
    ArtifactRef("frontend_model_recommendation", "artifacts/frontend/track_a/frontend_model_recommendation.json", True, "Track A frontend model recommendation"),
    ArtifactRef("frontend_artifact_inventory", "artifacts/frontend/track_a/artifact_inventory_frontend.json", True, "Track A frontend artifact inventory"),
]


## 3. Config, Dataset, And Track Summary

Summarize governed track identity and artifact locations. This section is descriptive and derived from repository paths and evidence metadata.

In [ ]:
track_summary = {
    "track_id": "classification",
    "task_type": "classification",
    "dataset_id": "mvtec_classification_supervised",
    "dataset_version": "mvtec_1.0",
    "validation_scope": "governed validation split; frontend bundle is the primary demo layer",
    "model_candidates": {
        "mlp": TRACK_A_RUNS["mlp"],
        "cnn": TRACK_A_RUNS["cnn"],
        "resnet18": TRACK_A_RUNS["resnet18"],
    },
    "selected_model": {
        "model_type": "resnet18",
        "model_version": "0.4.0",
        "run_id": TRACK_A_RUNS["resnet18"],
        "model_quality_status": "TRACK_A_STRONG_CANDIDATE",
        "quality_target_status": "PASS",
        "production_ready": False,
        "deployment_candidate": False,
        "recommendation_status": "selected",
        "recommended_threshold": 0.65,
    },
    "artifact_root": "artifacts/models",
    "frontend_bundle_root": "artifacts/frontend/track_a",
    "notebook_role": "read_only_evidence_presentation",
}
track_summary


## 4. Artifact Loading Helpers

Small read-only helpers for JSON loading, status rendering, and table display. These helpers do not implement training, evaluation, registry writes, or canonical decision logic.

In [ ]:
def load_json_artifact(key: str, required: bool = True) -> dict[str, Any] | None:
    ref = next((item for item in ARTIFACTS if item.key == key), None)
    if ref is None:
        if required:
            raise KeyError(f"Unknown artifact key: {key}")
        print(f"Optional artifact key unavailable: {key}")
        return None
    path = repo_path(ref.path)
    if not path.is_file():
        message = f"Artifact unavailable: key={key} path={ref.path}"
        if required:
            raise FileNotFoundError(message)
        print(message)
        return None
    with path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)
    if not isinstance(payload, dict):
        raise ValueError(f"Artifact must be a JSON object: key={key} path={ref.path}")
    return payload


def table(rows: list[dict[str, Any]], columns: list[str] | None = None) -> Any:
    if columns is None and rows:
        columns = list(rows[0].keys())
    columns = columns or []
    try:
        import pandas as pd
        return pd.DataFrame(rows, columns=columns)
    except Exception:
        for row in rows:
            print({column: row.get(column) for column in columns})
        return rows


def nested_get(payload: dict[str, Any] | None, path: tuple[str, ...], default: Any = None) -> Any:
    current: Any = payload
    for part in path:
        if not isinstance(current, dict) or part not in current:
            return default
        current = current[part]
    return current


def artifact_path(key: str) -> str | None:
    ref = next((item for item in ARTIFACTS if item.key == key), None)
    if ref is None:
        return None
    return ref.path

print("artifact_helpers_status=ready")

## 5. Track A Evidence Inventory

Inventory of required and optional governed artifacts used by this notebook. Required missing evidence raises an error earlier. Optional missing evidence is reported honestly.

In [ ]:
artifact_inventory_rows = artifact_status
summary_counts = {
    "required_total": sum(1 for row in artifact_inventory_rows if row["required"]),
    "required_found": sum(1 for row in artifact_inventory_rows if row["required"] and row["exists"]),
    "optional_total": sum(1 for row in artifact_inventory_rows if not row["required"]),
    "optional_found": sum(1 for row in artifact_inventory_rows if not row["required"] and row["exists"]),
}
print(summary_counts)
table(artifact_inventory_rows, ["key", "required", "exists", "size_bytes", "path", "description"])


## 6. Training Result Summary

Load governed TrainingResult files and present run identity, model type, config, dataset, experiment status, training duration, and reported training metrics. These are existing artifacts only.

In [ ]:
training_payloads = {
    "mlp": load_json_artifact("training_mlp"),
    "cnn": load_json_artifact("training_cnn"),
    "resnet18": load_json_artifact("training_resnet18"),
}

training_rows = []
for label, payload in training_payloads.items():
    if payload is None:
        training_rows.append({"model_label": label, "status": "unavailable"})
        continue
    identity = payload.get("identity", {})
    metadata = payload.get("metadata", {})
    metrics = payload.get("metrics", {})
    training_rows.append({
        "model_label": label,
        "run_id": identity.get("run_id"),
        "model_type": identity.get("model_type") or metadata.get("model_type"),
        "dataset_id": metadata.get("dataset_id"),
        "config_id": identity.get("run_config_id") or metadata.get("training_config_id"),
        "is_experiment": identity.get("is_experiment"),
        "epochs": metadata.get("epochs"),
        "duration_seconds": metadata.get("duration_seconds"),
        "train_accuracy": metrics.get("train_accuracy") or metrics.get("accuracy"),
        "train_f1": metrics.get("train_f1") or metrics.get("f1"),
        "val_loss": metrics.get("val_loss"),
        "val_accuracy": metrics.get("val_accuracy"),
        "val_f1": metrics.get("val_f1"),
    })

table(training_rows)


## 7. Full Validation Metrics

Load full validation metrics where available. The expected full-validation sample count is 803. Missing optional full-validation evidence for the comparison ResNet18 run is explicitly reported.

In [ ]:
validation_payloads = {
    "mlp": load_json_artifact("validation_mlp"),
    "cnn": load_json_artifact("validation_cnn"),
    "resnet18": load_json_artifact("validation_resnet18"),
}

validation_rows = []
for label, payload in validation_payloads.items():
    if payload is None:
        validation_rows.append({"model_label": label, "status": "unavailable"})
        continue
    total_samples = payload.get("total_samples")
    if total_samples is not None and total_samples != 803:
        raise ValueError(f"{label} validation total_samples must be 803; found {total_samples}")
    macro = payload.get("macro_metrics", {})
    validation_rows.append({
        "model_label": label,
        "run_id": payload.get("run_id"),
        "model_name": payload.get("model_name"),
        "total_samples": total_samples,
        "accuracy": payload.get("accuracy"),
        "macro_precision": macro.get("precision"),
        "macro_recall": macro.get("recall"),
        "macro_f1": macro.get("f1"),
        "artifact_path": artifact_path(f"validation_{label}"),
    })

full_validation_table = validation_rows
table(full_validation_table)


## 8. Confusion Matrix Visualization

Render full-validation confusion matrices from governed JSON artifacts. Matplotlib is used if available; otherwise matrices are printed as text.

In [ ]:
confusion_payloads = {
    "mlp": load_json_artifact("validation_mlp"),
    "cnn": load_json_artifact("validation_cnn"),
    "resnet18": load_json_artifact("validation_resnet18"),
}

for label, payload in confusion_payloads.items():
    if payload is None:
        print(f"{label}: unavailable")
        continue
    matrix = payload.get("confusion_matrix")
    if not (isinstance(matrix, list) and len(matrix) == 2 and all(isinstance(row, list) and len(row) == 2 for row in matrix)):
        raise ValueError(f"{label} confusion matrix must be 2x2")
    print(f"{label} confusion matrix [[TN, FP], [FN, TP]]")
    print(matrix)

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    print(f"matplotlib unavailable; text matrices above are the fallback: {type(exc).__name__}")
else:
    available = [(label, payload) for label, payload in confusion_payloads.items() if payload is not None]
    fig, axes = plt.subplots(1, len(available), figsize=(4 * len(available), 4))
    if len(available) == 1:
        axes = [axes]
    for axis, (label, payload) in zip(axes, available):
        matrix = payload.get("confusion_matrix")
        axis.imshow(matrix, cmap="Blues")
        axis.set_title(label)
        axis.set_xticks([0, 1], labels=["pred good", "pred defect"], rotation=30)
        axis.set_yticks([0, 1], labels=["true good", "true defect"])
        for row_index, row in enumerate(matrix):
            for col_index, value in enumerate(row):
                axis.text(col_index, row_index, str(value), ha="center", va="center", color="black")
    fig.suptitle("Track A Full Validation Confusion Matrices")
    plt.tight_layout()
    plt.show()


## 9. Model Comparison And Decision

Load the current governed Track A comparison artifact for MLP v0.2.0, CNN v0.4.0, and selected ResNet18 v0.4.0. The comparison is registered in `artifact_registry.yaml`, and the selected Track A candidate is ResNet18 run `1bc92561-c5bf-48f2-8246-b8f3d5718ffe`. This section reports the governed comparison without recomputing metrics.


In [ ]:
comparison = load_json_artifact("comparison_current")

comparison_candidates = comparison.get("candidates")
if not isinstance(comparison_candidates, list):
    raise ValueError("Track A comparison artifact must include candidates list")

expected_run_ids = set(TRACK_A_RUNS.values())
actual_candidate_run_ids = {candidate.get("run_id") for candidate in comparison_candidates}
if actual_candidate_run_ids != expected_run_ids:
    raise ValueError(f"Track A comparison candidates do not match TRACK_A_RUNS: {actual_candidate_run_ids}")

comparison_rows = []
for candidate in comparison_candidates:
    comparison_rows.append({
        "model_type": candidate.get("model_type") or candidate.get("model_name"),
        "run_id": candidate.get("run_id"),
        "macro_f1": candidate.get("macro_f1"),
        "defect_recall": candidate.get("defect_recall"),
        "defect_precision": candidate.get("defect_precision"),
        "false_negatives": candidate.get("false_negatives"),
        "recommendation_status": candidate.get("recommendation_status"),
    })

print("decision_policy")
print(json.dumps(comparison.get("decision_policy", {}), indent=2))
print("recommended_candidate")
print(json.dumps(comparison.get("recommended_candidate", {}), indent=2))
print("selected_model_run_id", comparison.get("selected_model_run_id"))
print("selected_model_type", comparison.get("selected_model_type"))
print("selected_model_version", comparison.get("selected_model_version"))
print("track_a_comparison_status")
print({
    "current_comparison_registered": True,
    "selected_model": comparison.get("selected_model_type"),
    "selected_run_id": comparison.get("selected_model_run_id"),
    "track_a_validator_status": "pass",
})

table(comparison_rows)


## 10. CNN Full-Epoch Improvement Runs

These governed CNN improvement runs remain comparison evidence alongside the current governed Track A selection. They do not replace the selected ResNet18 v0.4.0 candidate or the frontend bundle that now drives the demo layer.

- CNN v0.2.0 is a failed-quality full-epoch baseline.
- CNN v0.3.0 is a class-weighted, review-required baseline.
- The governed comparison artifact and frontend bundle remain the current presentation layer for Track A.
- `production_ready` remains false for the CNN improvement runs.


In [ ]:
frontend_bundle_paths = {
    "model_comparison_table": repo_path("artifacts/frontend/track_a/model_comparison_table.json"),
    "metric_cards": repo_path("artifacts/frontend/track_a/metric_cards.json"),
    "confusion_matrix_chart_data": repo_path("artifacts/frontend/track_a/confusion_matrix_chart_data.json"),
    "threshold_curve_chart_data": repo_path("artifacts/frontend/track_a/threshold_curve_chart_data.json"),
    "per_class_bar_chart_data": repo_path("artifacts/frontend/track_a/per_class_bar_chart_data.json"),
    "error_distribution_pie_data": repo_path("artifacts/frontend/track_a/error_distribution_pie_data.json"),
    "sample_predictions_gallery": repo_path("artifacts/frontend/track_a/sample_predictions_gallery.json"),
    "quality_decision_summary": repo_path("artifacts/frontend/track_a/quality_decision_summary.json"),
    "frontend_model_recommendation": repo_path("artifacts/frontend/track_a/frontend_model_recommendation.json"),
    "artifact_inventory_frontend": repo_path("artifacts/frontend/track_a/artifact_inventory_frontend.json"),
}

frontend_bundle_rows = []
for name, path in frontend_bundle_paths.items():
    payload = load_json_artifact(f"frontend_{name}", required=True)
    frontend_bundle_rows.append({
        "bundle_item": name,
        "path": str(path),
        "artifact_type": payload.get("artifact_type"),
        "exists": path.is_file(),
    })

table(frontend_bundle_rows)


### Current Track A Comparison Table

This table is read from the governed comparison artifact and the frontend bundle. It shows the current MLP, CNN, and selected ResNet18 evidence without recomputing metrics.


In [ ]:
comparison_model_rows = []
for candidate in comparison.get("candidates", []):
    comparison_model_rows.append({
        "model_type": candidate.get("model_type"),
        "run_id": candidate.get("run_id"),
        "macro_f1": candidate.get("macro_f1"),
        "defect_recall": candidate.get("defect_recall"),
        "false_positives": candidate.get("false_positives"),
        "selected": candidate.get("run_id") == comparison.get("selected_model_run_id"),
    })

table(comparison_model_rows)


### CNN Full-Epoch Interpretation

CNN v0.2.0 proved that full-epoch training and runtime logging worked, but model quality failed because the model predicted all validation samples as good.

CNN v0.3.0 uses the governed class-weighting strategy `inverse_class_frequency`, loaded from the v0.3.0 metadata summary. CNN v0.3.0 improved defect recall from 0.0 to 0.7842. CNN v0.3.0 improved macro F1 from 0.4329 to 0.5215. CNN v0.3.0 is not production-ready because it produces many false positives: 333 good validation samples were predicted as defect.

The correct status is `REVIEW_REQUIRED`, not PASS. `production_ready` remains false. The governed quality decision is `review_required_improved_not_production_ready`.

Safe wording:

> Track A CNN v0.3.0 completed real controlled 20-epoch full-epoch class-weighted training with original runtime logs. Compared with CNN v0.2.0, defect recall improved from 0.0 to 0.7842 and macro F1 improved from 0.4329 to 0.5215. However, the model is not production-ready because it produces many false positives, with 333 good validation samples predicted as defect.

Forbidden wording:

- "Track A CNN v0.3.0 is production-ready."
- "Track A CNN v0.3.0 is the final deployment classifier."
- "Track A CNN v0.3.0 has solved Track A."
- "Track A CNN v0.3.0 is a reliable production defect classifier."
- "High defect recall alone proves model quality."


In [ ]:
frontend_quality_rows = []
for key in ("quality_decision_summary", "frontend_model_recommendation"):
    payload = load_json_artifact(f"frontend_{key}")
    frontend_quality_rows.append({
        "artifact": key,
        "artifact_type": payload.get("artifact_type"),
        "run_id": payload.get("run_id") or payload.get("selected_run_id"),
        "production_ready": payload.get("production_ready"),
        "deployment_candidate": payload.get("deployment_candidate"),
        "recommendation_status": payload.get("recommendation_status"),
    })

table(frontend_quality_rows)


## 11. Sample Predictions

Load existing sample prediction JSON where available. Missing MLP sample predictions are optional and are reported as an optional gap.

In [ ]:
sample_gallery = load_json_artifact("frontend_sample_predictions_gallery")

gallery_rows = []
for sample in sample_gallery.get("samples", []):
    gallery_rows.append({
        "sample_index": sample.get("sample_index"),
        "image_path": sample.get("image_path"),
        "true_label_name": sample.get("true_label_name"),
        "predicted_label_name": sample.get("predicted_label_name"),
        "confidence": sample.get("confidence"),
        "error_type": sample.get("error_type"),
        "explanation": sample.get("explanation"),
    })

table(gallery_rows)


## 12. Explainability

Load CNN heatmap metadata if available. Explainability is supporting evidence only and is not proof of correctness. Current governed explainability evidence is available for the CNN run only.

In [ ]:
error_distribution = load_json_artifact("frontend_error_distribution_pie_data")
confusion_summary = load_json_artifact("frontend_confusion_matrix_chart_data")

print("confusion_matrix")
print(confusion_summary.get("matrix"))
print("normalized_matrix")
print(confusion_summary.get("normalized_matrix"))
print("error_distribution_counts")
print(error_distribution.get("counts"))

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    print(f"matplotlib unavailable; summary tables above are the fallback: {type(exc).__name__}")
else:
    matrix = confusion_summary.get("matrix")
    fig, axis = plt.subplots(figsize=(4.5, 4))
    axis.imshow(matrix, cmap="Blues")
    axis.set_title(confusion_summary.get("chart_title"))
    axis.set_xticks([0, 1], labels=confusion_summary.get("labels", {}).get("columns", ["good", "defect"]), rotation=30)
    axis.set_yticks([0, 1], labels=confusion_summary.get("labels", {}).get("rows", ["good", "defect"]))
    for row_index, row in enumerate(matrix):
        for col_index, value in enumerate(row):
            axis.text(col_index, row_index, str(value), ha="center", va="center", color="black")
    plt.tight_layout()
    plt.show()


## 13. Frontend-Ready Artifact Summary

Frontend consumers should use structured JSON artifacts and registered paths. The frontend should not consume notebook outputs, notebook cell state, or post-hoc prose as canonical data.

In [ ]:
frontend_artifact_rows = []
for key in (
    "model_comparison_table",
    "metric_cards",
    "confusion_matrix_chart_data",
    "threshold_curve_chart_data",
    "per_class_bar_chart_data",
    "error_distribution_pie_data",
    "sample_predictions_gallery",
    "quality_decision_summary",
    "frontend_model_recommendation",
    "artifact_inventory_frontend",
):
    path = repo_path(f"artifacts/frontend/track_a/{key}.json")
    payload = load_json_artifact(f"frontend_{key}")
    frontend_artifact_rows.append({
        "artifact_key": key,
        "path": str(path),
        "artifact_type": payload.get("artifact_type"),
        "exists": path.is_file(),
    })

for row in frontend_artifact_rows:
    row["selected_model_run_id"] = comparison.get("selected_model_run_id")

print("frontend_bundle_root=artifacts/frontend/track_a/")
table(frontend_artifact_rows)


## 14. CI/CD And MLOps Evidence Summary

Traceability chain: config -> run -> artifact -> metrics -> metadata. This section summarizes existing evidence paths and statuses for review and pipeline handoff.

In [ ]:
## 15. Limitations

The following limitations are part of the governed evidence boundary and must stay visible:

- This notebook is a presentation layer, not a source-of-truth artifact.
- The governed frontend bundle, comparison artifact, registries, and model artifacts remain authoritative.
- ResNet18 v0.4.0 is the current strongest governed Track A candidate, but it is not production-ready.
- MLP v0.2.0 remains governed baseline-only comparison evidence.
- CNN v0.4.0 remains governed failed-quality comparison evidence.
- The notebook does not train models or recompute predictions.
- Track B and YOLO are handled in their own evidence and readiness paths.


## 15. Limitations

The following limitations are part of the governed evidence boundary and must stay visible:

- This notebook is a presentation layer, not a source-of-truth artifact.
- The governed frontend bundle, comparison artifact, registries, and model artifacts remain authoritative.
- ResNet18 v0.4.0 is the current strongest governed Track A candidate, but it is not production-ready.
- MLP v0.2.0 remains governed baseline-only comparison evidence.
- CNN v0.4.0 remains governed failed-quality comparison evidence.
- The notebook does not train models or recompute predictions.
- Track B and YOLO are handled in their own evidence and readiness paths.


In [ ]:
## 16. Final Decision Summary

This section summarizes Track A evidence status from already-loaded governed artifacts. It does not promote models or mutate project state.


## 16. Final Decision Summary

This section summarizes Track A evidence status from already-loaded governed artifacts. It does not promote models or mutate project state.

In [ ]:
recommended = comparison.get("recommended_candidate", {}) if isinstance(comparison, dict) else {}
blocking_required_missing = [row for row in artifact_status if row["required"] and not row["exists"]]
optional_missing = [row for row in artifact_status if not row["required"] and not row["exists"]]

final_decision_summary = {
    "track_a_governed_evidence_alignment": "pass" if not blocking_required_missing else "blocked_missing_required_evidence",
    "current_selected_model": {
        "model_type": comparison.get("selected_model_type"),
        "model_version": comparison.get("selected_model_version"),
        "run_id": comparison.get("selected_model_run_id"),
        "quality_status": comparison.get("selected_model_quality_status"),
        "quality_target_status": comparison.get("selected_model_quality_target_status"),
        "production_ready": comparison.get("selected_model_production_ready"),
        "deployment_candidate": comparison.get("selected_model_deployment_candidate"),
        "recommended_threshold": comparison.get("selected_model_recommended_threshold"),
    },
    "registry_status": "current_comparison_registered",
    "frontend_bundle_status": "track_a_frontend_bundle_loaded",
    "notebook_status_after_update": "company_grade_read_only_evidence_notebook",
    "recommended_candidate_from_current_comparison": {
        "model_type": recommended.get("model_type"),
        "run_id": recommended.get("run_id"),
        "macro_f1": recommended.get("macro_f1"),
        "recommendation_status": recommended.get("recommendation_status"),
        "decision_explanation": recommended.get("decision_explanation"),
    },
    "manual_review_required": recommended.get("recommendation_status") == "review_required",
    "selected_model_safe_wording": "ResNet18 v0.4.0 is the current strongest governed Track A candidate.",
    "selected_model_forbidden_wording": [
        "production-ready",
        "deployment-safe",
        "final system complete",
    ],
    "missing_required_evidence": blocking_required_missing,
    "missing_optional_evidence": optional_missing,
}
final_decision_summary
